# Chapter 9 — Capstone: Text Classifier (Practice)

Work through these exercises **after reading** `notes/ch09-capstone-text-classifier.md`.

This capstone builds one real project end to end: a 3-class topic classifier (tech / sports / food) trained on the bundled `topic-classification-data.csv`. Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell**. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [ ]:
# ============================================================
# TOPIC: Capstone — assembling ch01-ch08 into one small, real text classifier
# MATH:  masked mean pool: pooled = sum(emb*mask)/sum(mask);  CE = -log softmax(logits)_true
# REF:   B00 ch09 notes — capstone-text-classifier
# ============================================================

# --- Imports ---
import csv
import math
import os
import random
import tempfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (this capstone is CPU-fast — 96 training examples)")

# --- Load the bundled dataset (ch09 notes §2) ---
CSV_PATH = Path("../topic-classification-data.csv")
with open(CSV_PATH, newline="") as f:
    all_rows = [(row["text"], row["label"]) for row in csv.DictReader(f)]

LABELS = ["tech", "sports", "food"]
label_to_id = {label: i for i, label in enumerate(LABELS)}

# a LOCAL Random instance (ch08 §7: isolate your own randomness, don't touch global state)
split_rng = random.Random(0)
shuffled_rows = all_rows.copy()
split_rng.shuffle(shuffled_rows)
split_point = int(len(shuffled_rows) * 0.8)
train_rows, val_rows = shuffled_rows[:split_point], shuffled_rows[split_point:]

print(f"\nloaded {len(all_rows)} rows -> train={len(train_rows)}, val={len(val_rows)}")
print(f"sample: {train_rows[0]}")

## Exercise 1 — Vocabulary From the Training Split Only

Implement `build_vocab(rows)`: a whitespace tokenizer's vocabulary, with reserved `<pad>`(id 0) and `<unk>`(id 1), built by scanning **only** `train_rows`. Then implement `encode(text, vocab)` mapping unknown words to `<unk>`.

**Decision you're practicing:** fitting any preprocessing statistic (a vocabulary, here) on train only — using validation data to build it would leak information from the set you're trying to honestly evaluate on.

In [ ]:
def build_vocab(rows):
    """Whitespace-tokenize every text in rows; return {word: id}, reserving
    <pad>=0 and <unk>=1, built ONLY from these rows."""
    # TODO
    pass

def encode(text, vocab):
    """Tokenize text on whitespace; map each word through vocab, unknown -> vocab['<unk>']."""
    # TODO
    pass

vocab = build_vocab(train_rows)
PAD_ID, UNK_ID = vocab["<pad>"], vocab["<unk>"]

**Verification**

In [ ]:
# --- Verification: Exercise 1 ---
assert vocab is not None, "fill in the stubs above first"
assert vocab["<pad>"] == 0 and vocab["<unk>"] == 1, "reserved ids must be exactly 0 and 1"
assert len(vocab) == 114, f"expected vocab size 114 (train-only), got {len(vocab)}"
assert vocab.get("the") == 2, "vocab ids must be assigned in first-seen order over train_rows"

# a made-up word must fall back to <unk>
made_up = encode("zzz_not_a_real_word", vocab)
assert made_up == [UNK_ID], f"unknown word should encode to [<unk>], got {made_up}"

# every word in train_rows must be encodable to a REAL id (never <unk>)
for text, _ in train_rows[:10]:
    ids = encode(text, vocab)
    assert UNK_ID not in ids, f"a training word fell back to <unk>: {text!r} -> {ids}"

print(f"vocab size {len(vocab)}, built from train only; <unk> fallback confirmed ✓")
print("Exercise 1 passed ✓")

## Exercise 2 — Dataset + collate_fn (ch06, for real this time)

Implement `TopicDataset` (map-style: `__len__`/`__getitem__`, returning **un-padded** `(token_ids, label_id)`) and `collate_batch` (pads with `pad_sequence`, builds the boolean mask, returns `(padded, mask, labels)`).

**Decision you're practicing:** the full ch06 pipeline, assembled — Dataset produces one raw example, collate turns a batch of them into padded rectangles plus a mask.

In [ ]:
class TopicDataset(Dataset):
    def __init__(self, rows, vocab):
        self.rows = rows
        self.vocab = vocab

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO: return (int64 token-id tensor, label_id) for ONE example
        pass

def collate_batch(examples):
    """examples: list of (token_ids, label_id).
    Returns (padded (B, max_len) int64, mask (B, max_len) bool, labels (B,))."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 2 ---
assert len(TopicDataset(train_rows, vocab)) == 96, "TopicDataset.__len__ must match train_rows"
item_ids, item_label = TopicDataset(train_rows, vocab)[0]
assert isinstance(item_ids, torch.Tensor) and item_ids.dtype == torch.long, "token ids must be int64"
assert isinstance(item_label, int), "label must be a plain int (the DataLoader will tensor-ify it)"

loader = DataLoader(TopicDataset(train_rows, vocab), batch_size=8, shuffle=False, collate_fn=collate_batch)
padded, mask, labels = next(iter(loader))
assert padded is not None, "fill in the stubs above first"
assert padded.dtype == torch.long and mask.dtype == torch.bool
assert padded.shape == mask.shape and padded.shape[0] == 8
assert torch.equal(mask, padded != PAD_ID), "mask must be exactly (padded != PAD_ID)"
assert labels.shape == (8,)

# no default-collate crash (ch06 §4): this dataset is ragged and MUST need collate_batch
try:
    DataLoader(TopicDataset(train_rows, vocab), batch_size=8, shuffle=False)  # no collate_fn
    default_batch = next(iter(DataLoader(TopicDataset(train_rows, vocab), batch_size=8)))
    raise AssertionError("default collate should have crashed on this ragged dataset")
except RuntimeError:
    pass
print(f"batch: padded {tuple(padded.shape)}, mask matches (padded != PAD_ID), default collate still crashes ✓")
print("Exercise 2 passed ✓")

## Exercise 3 — The Model: Embedding → Masked Mean-Pool → MLP

Implement `TextClassifier`: `nn.Embedding(padding_idx=PAD_ID)` → `masked_mean_pool` (as its own method, reused by exercise 7) → `Linear` → `GELU` → `Linear`, returning **raw logits**.

**Decision you're practicing:** composing registered submodules (ch04) correctly, masked pooling (ch01/ch06), and the raw-logits contract (ch05) — no softmax in `forward`.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_id):
        super().__init__()
        # TODO: self.embedding, self.fc1, self.fc2
        pass

    def masked_mean_pool(self, embedded, mask):
        """(B, T, D) embeddings + (B, T) bool mask -> (B, D), averaging REAL tokens only."""
        # TODO
        pass

    def forward(self, padded, mask):
        """padded: (B, T) int64, mask: (B, T) bool -> (B, num_classes) RAW LOGITS."""
        # TODO
        pass

**Verification**

In [ ]:
# --- Verification: Exercise 3 ---
torch.manual_seed(0)
test_model = TextClassifier(len(vocab), 24, 48, len(LABELS), PAD_ID)
param_count = sum(p.numel() for p in test_model.parameters())
assert param_count > 0, "zero parameters -- did a submodule fail to register? (ch04's plain-list trap)"
expected_count = (len(vocab) * 24) + (24 * 48 + 48) + (48 * 3 + 3)
assert param_count == expected_count, f"expected {expected_count} params, got {param_count}"

test_padded, test_mask, test_labels = next(iter(DataLoader(
    TopicDataset(train_rows, vocab), batch_size=4, shuffle=False, collate_fn=collate_batch)))
logits = test_model(test_padded, test_mask)
assert logits is not None, "fill in the stubs above first"
assert logits.shape == (4, 3), f"expected (4, 3) logits, got {tuple(logits.shape)}"
row_sums = logits.sum(dim=-1)
assert not torch.allclose(row_sums, torch.ones(4), atol=1e-3), \
    "each row sums to ~1 — did forward() accidentally apply softmax? logits should NOT be a distribution (ch05 gotcha)"

# gluing on extra padding must not change the pooled representation (ch06 exercise 7's proof, reused)
extra_pad = torch.full((4, 3), PAD_ID, dtype=torch.long)
wider_padded = torch.cat([test_padded, extra_pad], dim=1)
wider_mask = wider_padded != PAD_ID
embedded_narrow = test_model.embedding(test_padded)
embedded_wide = test_model.embedding(wider_padded)
pooled_narrow = test_model.masked_mean_pool(embedded_narrow, test_mask)
pooled_wide = test_model.masked_mean_pool(embedded_wide, wider_mask)
assert torch.allclose(pooled_narrow, pooled_wide, atol=1e-6), "extra padding changed the pooled vector!"
print(f"{param_count} parameters registered; logits are raw (unbounded); padding-invariance holds ✓")
print("Exercise 3 passed ✓")

## Exercise 4 — Optimizer Param Groups + Warmup/Cosine Schedule

Implement `build_param_groups` (decay 2-D+ weights, exempt 1-D biases — ch07 §5) and `warmup_cosine` (ch07 §6's formula). Wire both into a real `AdamW` + `LambdaLR`.

**Decision you're practicing:** the exact ch07 optimizer/schedule setup, applied to a real model instead of a toy tensor.

In [ ]:
def build_param_groups(model, weight_decay):
    """Split model.parameters() by ndim: >=2 gets weight_decay, 1-D gets 0.0."""
    # TODO
    pass

def warmup_cosine(step, warmup_steps, total_steps):
    """Linear warmup to a multiplier of 1.0, then cosine decay to 0.0."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 4 ---
groups = build_param_groups(model, weight_decay=0.01)
assert groups is not None, "fill in the stubs above first"
decay_group = next(g for g in groups if g["weight_decay"] == 0.01)
no_decay_group = next(g for g in groups if g["weight_decay"] == 0.0)
assert all(p.ndim >= 2 for p in decay_group["params"]), "decay group must be all 2-D+ (weight matrices)"
assert all(p.ndim == 1 for p in no_decay_group["params"]), "no-decay group must be all 1-D (biases)"
assert len(decay_group["params"]) == 3, "embedding.weight, fc1.weight, fc2.weight -> 3 tensors"
assert len(no_decay_group["params"]) == 2, "fc1.bias, fc2.bias -> 2 tensors"

checkpoints = [0, WARMUP_STEPS // 2, WARMUP_STEPS, WARMUP_STEPS + (TOTAL_STEPS - WARMUP_STEPS) // 2, TOTAL_STEPS]
lrs = [warmup_cosine(s, WARMUP_STEPS, TOTAL_STEPS) for s in checkpoints]
assert lrs[0] == 0.0, "step 0 should be lr multiplier 0.0"
assert abs(lrs[2] - 1.0) < 1e-9, "end of warmup should hit exactly multiplier 1.0"
assert abs(lrs[3] - 0.5) < 1e-6, "the schedule's halfway point should be multiplier 0.5"
assert lrs[4] < 1e-6, "end of training should decay to ~0"
print(f"param groups correctly split (3 decay, 2 no-decay); schedule shape confirmed at 5 checkpoints ✓")
print("Exercise 4 passed ✓")

## Exercise 5 — The Training Loop, End to End

Implement `train_one_epoch` (the ch07 seven-step order: zero_grad → forward → loss → backward → clip → step → schedule, run once per batch) and `evaluate` (`model.eval()` + `torch.no_grad()`, accuracy over a loader). Then train for `NUM_EPOCHS` and beat the majority-class baseline decisively.

**Decision you're practicing:** the entire ch07 loop, assembled around a real DataLoader instead of one fixed batch.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, max_norm=1.0):
    """One epoch. Returns the LAST batch's loss (a float)."""
    # TODO: model.train(); for each batch, run the 7-step loop
    pass

def evaluate(model, loader):
    """Accuracy over the whole loader, in eval mode, with no_grad."""
    # TODO
    pass

majority_baseline = max(sum(1 for _, l in val_rows if l == label) for label in LABELS) / len(val_rows)
print(f"majority-class baseline on val: {majority_baseline:.3f}")

**Verification**

In [ ]:
# --- Verification: Exercise 5 ---
assert train_one_epoch is not None and evaluate is not None, "fill in the stubs above first"

torch.manual_seed(7)
verify_model = TextClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), PAD_ID)
verify_optimizer = torch.optim.AdamW(build_param_groups(verify_model, weight_decay=0.01), lr=2e-2)
verify_scheduler = torch.optim.lr_scheduler.LambdaLR(
    verify_optimizer, lr_lambda=lambda step: warmup_cosine(step, WARMUP_STEPS, TOTAL_STEPS)
)
for epoch in range(NUM_EPOCHS):
    train_one_epoch(verify_model, train_loader, verify_optimizer, verify_scheduler)

val_acc = evaluate(verify_model, val_loader)
assert val_acc > majority_baseline + 0.3, \
    f"val acc {val_acc:.3f} should decisively beat the {majority_baseline:.3f} baseline"
assert verify_scheduler.last_epoch == TOTAL_STEPS, \
    f"scheduler should have stepped {TOTAL_STEPS} times, stepped {verify_scheduler.last_epoch}"

# train() vs eval() actually matters here too -- confirm eval mode is deterministic
verify_model.eval()
with torch.no_grad():
    p, m, l = next(iter(val_loader))
    out_a = verify_model(p, m)
    out_b = verify_model(p, m)
assert torch.equal(out_a, out_b), "eval-mode forward passes on the same input must be identical"
print(f"trained {NUM_EPOCHS} epochs -> val acc {val_acc:.3f} (baseline {majority_baseline:.3f}) ✓")
print("Exercise 5 passed ✓")

## Exercise 6 — Checkpoint That Actually Resumes

Save an honest checkpoint (model + optimizer + scheduler + epoch) partway through training, then resume into fresh objects and confirm training continues smoothly — the optimizer's Adam momentum and the scheduler's position on the warmup/cosine curve must both survive exactly.

**Decision you're practicing:** ch07's checkpoint lesson, now on the real model — a model-only save would restart Adam's momentum from zero and visibly stumble on resume.

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, path):
    # TODO
    pass

def load_checkpoint(model, optimizer, scheduler, path):
    """Load in place; return the saved epoch."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 6 ---
CKPT_PATH = os.path.join(tempfile.gettempdir(), "b00_ch09_capstone_ckpt_verify.pt")
HALFWAY = NUM_EPOCHS // 2

torch.manual_seed(13)
model_a = TextClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), PAD_ID)
opt_a = torch.optim.AdamW(build_param_groups(model_a, weight_decay=0.01), lr=2e-2)
sched_a = torch.optim.lr_scheduler.LambdaLR(opt_a, lr_lambda=lambda s: warmup_cosine(s, WARMUP_STEPS, TOTAL_STEPS))
for epoch in range(HALFWAY):
    train_one_epoch(model_a, train_loader, opt_a, sched_a)
save_checkpoint(model_a, opt_a, sched_a, HALFWAY - 1, CKPT_PATH)
assert os.path.exists(CKPT_PATH), "save_checkpoint did not write a file"
momentum_at_save = opt_a.state[model_a.fc1.weight]["exp_avg"].clone()

model_b = TextClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), PAD_ID)
opt_b = torch.optim.AdamW(build_param_groups(model_b, weight_decay=0.01), lr=2e-2)
sched_b = torch.optim.lr_scheduler.LambdaLR(opt_b, lr_lambda=lambda s: warmup_cosine(s, WARMUP_STEPS, TOTAL_STEPS))
resumed_epoch = load_checkpoint(model_b, opt_b, sched_b, CKPT_PATH)
assert resumed_epoch == HALFWAY - 1, f"resumed epoch should be {HALFWAY - 1}, got {resumed_epoch}"
assert torch.equal(model_a.fc1.weight, model_b.fc1.weight), "weights must match after resume"
momentum_after_load = opt_b.state[model_b.fc1.weight]["exp_avg"]
assert torch.equal(momentum_at_save, momentum_after_load), "Adam momentum must survive the checkpoint exactly"
assert sched_b.last_epoch == sched_a.last_epoch, "scheduler position must match after resume"

# continuing from model_b should match continuing from model_a directly (same optimizer/scheduler state)
for _ in range(3):
    train_one_epoch(model_a, train_loader, opt_a, sched_a)
    train_one_epoch(model_b, train_loader, opt_b, sched_b)
assert torch.allclose(model_a.fc1.weight, model_b.fc1.weight, atol=1e-5), \
    "continuing after an honest resume should track the un-interrupted run"
print("Adam momentum and scheduler position survived the checkpoint; resumed training tracks the original run ✓")
print("Exercise 6 passed ✓")

## Exercise 7 — The Debugging Challenge

`BuggyTextClassifier` below is architecturally identical to yours and trains the same way — it does **not crash**, and its `train_loss` looks like a real number. But its validation accuracy will land at or below chance. Per notes §5: **don't just re-read the code**. Implement `diagnose_pooling_bug(model, dataset)`: run `pool_only` on two *different* examples and compare the results — if the pooled vectors are (almost) identical regardless of input, the model has stopped seeing its input somewhere before that point.

Then implement `FixedTextClassifier` — identical to `BuggyTextClassifier` except for the one corrected line — and confirm it trains to high accuracy.

**Decision you're practicing:** the ch08 method — instrument, check an invariant, let the symptom point you at the cause — applied to a correctness bug instead of a crash.

In [ ]:
class BuggyTextClassifier(nn.Module):
    """Looks right. Trains. Never gets good. Find the one broken line."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def pool_only(self, padded, mask):
        embedded = self.embedding(padded)
        mask_float = (~mask).unsqueeze(-1).to(embedded.dtype)     # <-- somewhere near here...
        summed = (embedded * mask_float).sum(dim=1)
        counts = mask_float.sum(dim=1).clamp(min=1)
        return summed / counts

    def forward(self, padded, mask):
        pooled = self.pool_only(padded, mask)
        return self.fc2(F.gelu(self.fc1(pooled)))

def diagnose_pooling_bug(model, dataset):
    """Compare pool_only() output for dataset[0] and dataset[1] (batched via collate_batch).
    Return the max absolute difference between the two pooled vectors."""
    # TODO
    pass

# TODO: identical to BuggyTextClassifier, with the ONE line fixed
class FixedTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_id):
        super().__init__()
        pass

    def pool_only(self, padded, mask):
        pass

    def forward(self, padded, mask):
        pass

**Verification**

In [ ]:
# --- Verification: Exercise 7 ---
torch.manual_seed(7)
buggy_model = BuggyTextClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), PAD_ID)
buggy_diff = diagnose_pooling_bug(buggy_model, TopicDataset(val_rows, vocab))
assert buggy_diff is not None, "implement diagnose_pooling_bug first"
assert buggy_diff < 1e-4, f"expected the buggy model's pooled vectors to be ~identical, diff={buggy_diff}"

torch.manual_seed(7)
fixed_model = FixedTextClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), PAD_ID)
fixed_diff = diagnose_pooling_bug(fixed_model, TopicDataset(val_rows, vocab))
assert fixed_diff > 0.1, f"the fixed model should clearly distinguish different sentences, diff={fixed_diff}"

# train BOTH end to end and confirm the accuracy gap the diagnosis predicted
def full_training_run(model_cls):
    torch.manual_seed(7)
    m = model_cls(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), PAD_ID)
    opt = torch.optim.AdamW(build_param_groups(m, weight_decay=0.01), lr=2e-2)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda s: warmup_cosine(s, WARMUP_STEPS, TOTAL_STEPS))
    for _ in range(NUM_EPOCHS):
        train_one_epoch(m, train_loader, opt, sch)
    return evaluate(m, val_loader)

buggy_val_acc = full_training_run(BuggyTextClassifier)
fixed_val_acc = full_training_run(FixedTextClassifier)
assert buggy_val_acc <= majority_baseline + 0.05, \
    f"buggy model should NOT beat the baseline meaningfully, got {buggy_val_acc:.3f}"
assert fixed_val_acc > majority_baseline + 0.3, \
    f"fixed model should decisively beat the baseline, got {fixed_val_acc:.3f}"
print(f"buggy val acc: {buggy_val_acc:.3f} (~baseline)   fixed val acc: {fixed_val_acc:.3f}  ✓")
print("Exercise 7 passed ✓")

---
## Done!

Compare your work against `solved/ch09-capstone-text-classifier-solved.ipynb`.

**That completes B00 — PyTorch Mastery for LLMs.** Nine chapters, every decision guide exercised at least once, ending here: a real, small, fully-understood text classifier, including the one skill no chapter alone could teach — finding a bug that doesn't announce itself, by checking whether the code's actual behavior matches an invariant it's supposed to satisfy.